In [10]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_groq import ChatGroq

from langgraph.checkpoint.memory import MemorySaver

In [11]:
from dotenv import load_dotenv

load_dotenv()

True

In [12]:
from langgraph.graph.message import add_messages

class ChatState(TypedDict):

    messages: Annotated[list[BaseMessage], add_messages]

In [13]:
llm = ChatGroq(model = "openai/gpt-oss-120b")


def chat_node(state: ChatState):

    # take user query from state
    messages = state['messages']

    # send to llm
    response = llm.invoke(messages)

    # response store state
    return {'messages': [response]}

In [14]:
checkpointer = MemorySaver()
graph = StateGraph(ChatState)

# add nodes
graph.add_node('chat_node', chat_node)

graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

chatbot = graph.compile(checkpointer = checkpointer)

In [15]:
thread_id = "1"
while True:
    user_msg = input("Ask anything!!")
    print("User :", user_msg)
    if user_msg.strip().lower() in ["exit", "bye", "quit"]:
        break
    config = {"configurable" : {"thread_id" : thread_id}}
    response = chatbot.invoke({"messages" : [HumanMessage(content = user_msg)]}, config = config)
    print("AI :", response["messages"] [-1].content)


User : hi my name is aman
AI : Hi Aman! 👋 Nice to meet you. How can I help you today?
User : what is my name
AI : Your name is Aman. 🌟
User : what is 2 plus 2
AI : 2 + 2 = 4.
User : now divide the result by 2
AI : 4 ÷ 2 = 2.
User : bye


In [16]:
chatbot.get_state(config=config)

StateSnapshot(values={'messages': [HumanMessage(content='hi my name is aman', additional_kwargs={}, response_metadata={}, id='b92fef95-e42e-47e1-b8af-fdadb91394ec'), AIMessage(content='Hi Aman! 👋 Nice to meet you. How can I help you today?', additional_kwargs={'reasoning_content': 'We need to greet and maybe ask about what they need. The user said hi, name is Aman. So respond friendly.'}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 76, 'total_tokens': 128, 'completion_time': 0.109807841, 'completion_tokens_details': {'reasoning_tokens': 26}, 'prompt_time': 0.003866007, 'prompt_tokens_details': None, 'queue_time': 0.394337918, 'total_time': 0.113673848}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_803c0ba83d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a11c3b-72ca-74a3-8f20-10df7561aa7d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 76, 'ou